In [ ]:
import asyncio
from claude_agent_sdk import ClaudeSDKClient, ClaudeAgentOptions, AssistantMessage, ResultMessage, ToolUseBlock

import nest_asyncio
import os 
import pandas as pd
import os
from pathlib import Path
dataset_path = Path("~/mnt/shared-resources-sentient-research/salah_resources/datasets/officeqa/")
csv_path = os.path.join(dataset_path, "officeqa.csv")
file_paths = Path("../transformed/")
system_prompt = {"type": "preset", "preset": "claude_code", "append": "You are an expert analyst working at a top office firm. You are given a question and a set of documents. Your job is to answer the question based on the documents. You are also given a set of tools that you can use to answer the question. You can use the tools to read the documents, write the answer, or run code."}
available_tools = ["Read", "Write", "Bash", "Glob", "Grep", "Edit", "WebFetch", "WebSearch", "TodoWrite", "BashOutput"]

cwd = os.getcwd()

In [2]:
train_data = pd.read_csv(csv_path)
hard_data = train_data[train_data['difficulty'] == 'hard']
easy_data = train_data[train_data['difficulty'] == 'easy']


In [3]:
sample_question = hard_data.sample(1, random_state=42)
query = sample_question.question.values[0]
answer = sample_question.answer.values[0]
difficulty = sample_question.difficulty.values[0]

In [4]:
async def main():
    options = ClaudeAgentOptions(
        system_prompt=system_prompt,
        allowed_tools=available_tools,
        permission_mode='acceptEdits',
        add_dirs=[file_paths],
        cwd=cwd
    )

    intermediate_messages = []
    final_result = None

    async with ClaudeSDKClient(options=options) as client:
        await client.query(query)

        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                final_result = message
            else:
                intermediate_messages.append(message)
            print(message)

    return {
        "intermediate_messages": intermediate_messages,
        "final_result": final_result,
        "result_text": final_result.result if final_result else None,
        "is_error": final_result.is_error if final_result else None,
        "total_cost_usd": final_result.total_cost_usd if final_result else None,
        "num_turns": final_result.num_turns if final_result else None,
        "session_id": final_result.session_id if final_result else None
    }

In [5]:
nest_asyncio.apply()
result = asyncio.run(main())

SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': '/Users/salahalzubi/cursor_projects/deep_agents', 'session_id': '091a57c4-27f4-4e55-bcbb-7008af15563e', 'tools': ['Task', 'TaskOutput', 'Bash', 'Glob', 'Grep', 'ExitPlanMode', 'Read', 'Edit', 'Write', 'NotebookEdit', 'WebFetch', 'TodoWrite', 'WebSearch', 'KillShell', 'Skill', 'SlashCommand', 'EnterPlanMode'], 'mcp_servers': [], 'model': 'claude-opus-4-5-20251101', 'permissionMode': 'acceptEdits', 'slash_commands': ['compact', 'context', 'cost', 'init', 'pr-comments', 'release-notes', 'review', 'security-review'], 'apiKeySource': 'none', 'claude_code_version': '2.0.68', 'output_style': 'default', 'agents': ['general-purpose', 'statusline-setup', 'Explore', 'Plan'], 'skills': [], 'plugins': [], 'uuid': 'fa63c3d3-494e-4245-92eb-d8ad0b77f504'})
AssistantMessage(content=[ToolUseBlock(id='toolu_01FEYtwdhvpbvTiP8uy5hKDR', name='Task', input={'description': 'Find DOL budget data', 'prompt': 'Search the codebase for

In [6]:
print(result['final_result'].result)

Based on the U.S. Department of Labor's total outlays from the Treasury Bulletin data (Table FFO-4 which includes both budgetary and trust-fund flows):

- **FY 2011**: $131,973 million
- **FY 2019**: $35,810 million

The calculations yield:

1. **CAGR**: (35,810 / 131,973)^(1/8) - 1 = **-0.150**
2. **Annual Decay Factor**: (35,810 / 131,973)^(1/8) = **0.850**
3. **Arc Elasticity** (midpoint method): [(35,810 - 131,973) / ((35,810 + 131,973)/2)] / [(2019 - 2011) / ((2019 + 2011)/2)] = **-288.719**

**[-0.150, 0.850, -288.719]**


'[-0.153, 0.847, -1.162]'